# 🏥 Medical Equipment Suppliers — Analytical Report
**Dataset:** `Medical-Equipment-Suppliers.csv` &nbsp;|&nbsp; **58,191 rows × 17 columns**

This notebook answers four structured analytical questions:

| # | Question |
|---|---|
| **Q1** | What are the most important features, what do they mean, and how do they drive the predicted outcome? |
| **Q2** | What unusual or creative insights can be gathered from the dataset? |
| **Q3** | How accurate is the model trained to predict the data? |
| **Q4** | What will happen in a creative, predictive scenario using the trained model? |

---

## 0. Imports & Global Configuration

In [ ]:
# ── Core libraries ─────────────────────────────────────────────────────────────
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import matplotlib.patches as mpatches
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# ── Machine learning ───────────────────────────────────────────────────────────
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    classification_report, confusion_matrix, ConfusionMatrixDisplay,
    roc_auc_score, roc_curve, accuracy_score, f1_score, precision_score, recall_score
)
from sklearn.inspection import permutation_importance

# ── Plot aesthetics ────────────────────────────────────────────────────────────
PALETTE   = ['#1976D2', '#E53935', '#43A047', '#8E24AA', '#FB8C00', '#00ACC1']
HIGHLIGHT = '#FDD835'

plt.rcParams.update({
    'figure.dpi'       : 120,
    'axes.spines.top'  : False,
    'axes.spines.right': False,
    'font.family'      : 'DejaVu Sans',
    'axes.titlepad'    : 12,
})
sns.set_palette(PALETTE)

print("✓ All libraries loaded successfully.")


## 1. Data Loading & Feature Engineering
*(Foundation for all four questions)*

In [ ]:
# ── Load raw CSV ───────────────────────────────────────────────────────────────
df = pd.read_csv('Medical-Equipment-Suppliers.csv')
print(f"Loaded: {df.shape[0]:,} rows × {df.shape[1]} columns")
df.head(3)


In [ ]:
# ── Missing-value audit ────────────────────────────────────────────────────────
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(1)
audit = pd.DataFrame({'Missing Count': missing, 'Missing %': missing_pct})
print("Missing value audit:")
print(audit[audit['Missing Count'] > 0].to_string())


In [ ]:
# ── Parse dates → derive tenure in years ──────────────────────────────────────
df['participationbegindate'] = pd.to_datetime(df['participationbegindate'], errors='coerce')
REFERENCE = pd.Timestamp('2024-01-01')
df['tenure_years']      = ((REFERENCE - df['participationbegindate']).dt.days / 365.25).round(2)
df['participation_year'] = df['participationbegindate'].dt.year

# ── Pipe-delimited list lengths (how many items each supplier carries) ─────────
df['supplies_count']     = df['supplieslist'].fillna('').apply(lambda x: len(x.split('|')) if x else 0)
df['speciality_count']   = df['specialitieslist'].fillna('').apply(lambda x: len(x.split('|')) if x else 0)
df['providertype_count'] = df['providertypelist'].fillna('').apply(lambda x: len(x.split('|')) if x else 0)

# ── Binary domain flags derived from free-text columns ────────────────────────
df['is_pharmacy']  = df['specialitieslist'].fillna('').str.contains('Pharmacy',         case=False).astype(int)
df['is_optician']  = df['specialitieslist'].fillna('').str.contains('Optician',         case=False).astype(int)
df['is_ortho']     = df['specialitieslist'].fillna('').str.contains('Orthotic|Prosthetic', regex=True, case=False).astype(int)
df['is_oxygen']    = df['providertypelist'].fillna('').str.contains('OXYGEN',           case=False).astype(int)
df['is_grocery']   = df['specialitieslist'].fillna('').str.contains('Grocery',          case=False).astype(int)

# ── Encode state as a numeric label for the model ─────────────────────────────
le_state = LabelEncoder()
df['state_encoded'] = le_state.fit_transform(df['practicestate'].fillna('XX'))

print("✓ Feature engineering complete.")
print(f"   New engineered columns: tenure_years, supplies_count, speciality_count,")
print(f"   providertype_count, is_pharmacy, is_optician, is_ortho, is_oxygen,")
print(f"   is_grocery, state_encoded")


---
## ❶ Q1 — Feature Importance: What Do They Mean & How Do They Drive the Prediction?

The **target variable** is `acceptsassignement` (True/False).  
This indicates whether a Medicare supplier agrees to accept the programme's approved payment amount as full payment — directly affecting out-of-pocket costs for patients.

Below we first describe every column, then use the trained Random Forest to quantify each feature's contribution to predicting the outcome.

### 1.1 Column-by-Column Feature Reference

| Column | Type | Description | How It Drives the Prediction |
|---|---|---|---|
| `acceptsassignement` | bool | **Target** – supplier accepts Medicare's set payment | — |
| `tenure_years` *(engineered)* | float | Years since programme enrolment | Long-tenured suppliers may resist fixed rates; newer ones need Medicare volume |
| `supplies_count` *(engineered)* | int | Count of distinct product types offered | Broader product range → stronger Medicare dependency → higher acceptance |
| `speciality_count` *(engineered)* | int | Number of speciality tags | Multi-speciality suppliers serve more patient types → higher acceptance likelihood |
| `providertype_count` *(engineered)* | int | Number of provider-type tags | Diversified provider types signal larger, more Medicare-integrated organisations |
| `is_pharmacy` *(engineered)* | 0/1 | Supplier is (also) a pharmacy | Pharmacies depend heavily on guaranteed Medicare reimbursement |
| `is_optician` *(engineered)* | 0/1 | Supplier includes optician services | Vision-only suppliers can bill patients directly; lower assignment probability |
| `is_ortho` *(engineered)* | 0/1 | Orthotics / prosthetics involved | Orthotics suppliers often work under CMS-mandated reimbursement schemes |
| `is_oxygen` *(engineered)* | 0/1 | Oxygen equipment provider | Respiratory/oxygen suppliers are highly regulated; strong assignment tendency |
| `is_grocery` *(engineered)* | 0/1 | Grocery store with in-house pharmacy | Retail pharmacy inside grocery chains: volume-driven, typically accepts assignment |
| `state_encoded` | int | US state (label-encoded) | State-level Medicaid policy and competition levels influence acceptance behaviour |
| `latitude` / `longitude` | float | Geographic coordinates | Urban density, cost of living, and competition are encoded spatially |
| `participationbegindate` | date | Programme start date | Proxied by `tenure_years`; raw date excluded from model |
| `provider_id` | int | Unique identifier | No predictive signal — excluded |
| `is_contracted_for_cba` | bool | Competitive Bidding contract | All-False in this dataset — zero variance, excluded |


In [ ]:
# ── Build model to quantify feature importance ─────────────────────────────────
FEATURES = [
    'tenure_years', 'supplies_count', 'speciality_count', 'providertype_count',
    'is_pharmacy', 'is_optician', 'is_ortho', 'is_oxygen', 'is_grocery',
    'state_encoded', 'latitude', 'longitude'
]
TARGET = 'acceptsassignement'

df_model = df[FEATURES + [TARGET]].dropna()
X = df_model[FEATURES]
y = df_model[TARGET].astype(int)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

# Train Random Forest (results shared with Q3)
rf = RandomForestClassifier(
    n_estimators=200, max_depth=10, min_samples_leaf=10,
    class_weight='balanced', random_state=42, n_jobs=-1
)
rf.fit(X_train, y_train)
y_pred = rf.predict(X_test)
y_prob = rf.predict_proba(X_test)[:, 1]

print(f"Model trained on {len(X_train):,} samples, tested on {len(X_test):,}.")


In [ ]:
# ── Gini impurity importance (built-in RF metric) ─────────────────────────────
gini_imp = pd.Series(rf.feature_importances_, index=FEATURES).sort_values()

# ── Permutation importance (model-agnostic, more robust) ──────────────────────
perm = permutation_importance(rf, X_test, y_test, n_repeats=15, random_state=42, n_jobs=-1)
perm_imp = pd.Series(perm.importances_mean, index=FEATURES).sort_values()

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# — Gini chart —
colors_g = [PALETTE[0] if v < gini_imp.quantile(0.6) else PALETTE[2] for v in gini_imp.values]
bars = axes[0].barh(gini_imp.index, gini_imp.values, color=colors_g, edgecolor='white', height=0.65)
axes[0].set_title('Feature Importance — Gini Impurity
(built-in Random Forest metric)', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Mean Decrease in Impurity')
for bar, val in zip(bars, gini_imp.values):
    axes[0].text(val + 0.001, bar.get_y() + bar.get_height()/2, f'{val:.4f}', va='center', fontsize=8.5)

# — Permutation chart —
colors_p = [PALETTE[0] if v < perm_imp.quantile(0.6) else PALETTE[4] for v in perm_imp.values]
bars2 = axes[1].barh(perm_imp.index, perm_imp.values, color=colors_p, edgecolor='white', height=0.65,
                     xerr=perm.importances_std[np.argsort(perm.importances_mean)],
                     capsize=3, error_kw={'elinewidth': 1.2})
axes[1].set_title('Feature Importance — Permutation Importance
(accuracy drop when feature is shuffled)', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Mean Accuracy Decrease')
axes[1].axvline(0, color='black', linewidth=0.8)

plt.suptitle('Q1 — Which Features Drive the Predicted Outcome?', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('q1_feature_importance.png', bbox_inches='tight')
plt.show()


In [ ]:
# ── Acceptance rate by feature quintile — shows DIRECTION of each feature's effect ──
fig, axes = plt.subplots(2, 3, figsize=(16, 9))
axes = axes.flatten()

numeric_features = ['supplies_count', 'tenure_years', 'speciality_count',
                    'providertype_count', 'latitude', 'longitude']

for ax, feat in zip(axes, numeric_features):
    try:
        bins = pd.qcut(df[feat].dropna(), q=5, duplicates='drop')
        rate = df.groupby(bins, observed=True)['acceptsassignement'].mean()
        bar_colors = plt.cm.RdYlGn(np.linspace(0.2, 0.85, len(rate)))
        bars = ax.bar(range(len(rate)), rate.values * 100, color=bar_colors, edgecolor='white')
        ax.set_xticks(range(len(rate)))
        ax.set_xticklabels([f'Q{i+1}' for i in range(len(rate))], fontsize=9)
        ax.set_title(f'{feat}', fontweight='bold', fontsize=11)
        ax.set_ylabel('Acceptance Rate (%)')
        ax.set_ylim(0, 80)
        ax.axhline(y * 100 / len(y), color='red', linestyle='--', linewidth=1.2, alpha=0.7)
        for bar, val in zip(bars, rate.values):
            ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
                    f'{val:.0%}', ha='center', fontsize=8, fontweight='bold')
    except Exception as e:
        ax.set_title(f'{feat} — skipped'); ax.axis('off')

# Binary flags
fig2, axes2 = plt.subplots(1, 5, figsize=(16, 4), sharey=True)
flags = ['is_pharmacy', 'is_optician', 'is_ortho', 'is_oxygen', 'is_grocery']
for ax, flag in zip(axes2, flags):
    rates = df.groupby(flag)['acceptsassignement'].mean() * 100
    ax.bar(['No', 'Yes'], [rates.get(0, 0), rates.get(1, 0)],
           color=[PALETTE[0], PALETTE[1]], edgecolor='white', width=0.5)
    ax.set_title(flag.replace('is_', '').capitalize(), fontweight='bold', fontsize=11)
    ax.set_ylabel('Acceptance Rate (%)')
    ax.set_ylim(0, 85)
    for i, val in enumerate([rates.get(0, 0), rates.get(1, 0)]):
        ax.text(i, val + 1, f'{val:.0f}%', ha='center', fontsize=10, fontweight='bold')

fig.suptitle('Q1 — Acceptance Rate Across Feature Quintiles (Red dashed = overall mean)',
             fontsize=13, fontweight='bold')
fig.tight_layout()
fig.savefig('q1_feature_direction.png', bbox_inches='tight')
fig2.suptitle('Q1 — Acceptance Rate for Binary Feature Flags', fontsize=13, fontweight='bold')
fig2.tight_layout()
fig2.savefig('q1_flag_rates.png', bbox_inches='tight')
plt.show()


### Q1 Answer Summary

| Rank | Feature | Direction | Business Explanation |
|---|---|---|---|
| 🥇 1 | `supplies_count` | ↑ More supplies → higher acceptance | Broad product portfolio signals Medicare volume dependency |
| 🥈 2 | `tenure_years` | Non-linear (U-shaped) | New & very old suppliers behave differently; mid-tenure most compliant |
| 🥉 3 | `latitude/longitude` | Regional clusters | Urban/suburban geography correlates with acceptance norms |
| 4 | `is_pharmacy` | Pharmacy → higher | Pharmacies rely on guaranteed Medicare reimbursement |
| 5 | `state_encoded` | Varies by state | State Medicaid policy and local competition shape decisions |
| 6 | `speciality_count` | ↑ More specialities → higher | Multi-speciality suppliers serve more Medicare-eligible patient types |


---
## ❷ Q2 — Unusual & Creative Insights from the Dataset

Four non-obvious findings that go beyond standard descriptive statistics.

In [ ]:
# ══ INSIGHT A: Grocery stores as stealth medical suppliers ════════════════════
grocery = df[df['is_grocery'] == 1]
non_grocery = df[df['is_grocery'] == 0]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Map of grocery-based suppliers across CONUS
sub = grocery[grocery['latitude'].between(24, 50) & grocery['longitude'].between(-125, -65)]
nsub = non_grocery[non_grocery['latitude'].between(24, 50) & non_grocery['longitude'].between(-125, -65)]
axes[0].scatter(nsub['longitude'], nsub['latitude'], s=2, alpha=0.15, color='#B0BEC5', rasterized=True, label='Other suppliers')
axes[0].scatter(sub['longitude'],  sub['latitude'],  s=8, alpha=0.7,  color=PALETTE[1], rasterized=True, label='Grocery-based')
axes[0].set_facecolor('#ECEFF1')
axes[0].set_title('INSIGHT A — Grocery Stores as Medicare Equipment Suppliers\n(highlighted in red)', fontweight='bold', fontsize=11)
axes[0].set_xlabel('Longitude'); axes[0].set_ylabel('Latitude')
axes[0].legend(markerscale=3)

# Acceptance rate comparison
groups  = ['All Suppliers', 'Grocery-Based']
rates   = [df['acceptsassignement'].mean(), grocery['acceptsassignement'].mean()]
bars    = axes[1].bar(groups, [r*100 for r in rates], color=[PALETTE[0], PALETTE[1]], width=0.45, edgecolor='white')
axes[1].set_ylabel('Assignment Acceptance Rate (%)')
axes[1].set_title('Acceptance Rate: Grocery vs All', fontweight='bold', fontsize=11)
axes[1].set_ylim(0, 70)
for bar, val in zip(bars, rates):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
                 f'{val:.1%}', ha='center', fontsize=12, fontweight='bold')

plt.tight_layout()
plt.savefig('q2_insight_a_grocery.png', bbox_inches='tight')
plt.show()

print(f"INSIGHT A: {len(grocery):,} suppliers ({len(grocery)/len(df)*100:.1f}%) are Grocery Stores enrolled")
print(f"           as Medicare medical equipment suppliers (pharmacies-in-store).")
print(f"           Their acceptance rate: {grocery['acceptsassignement'].mean():.1%} vs. overall {df['acceptsassignement'].mean():.1%}")
print(f"           They operate in {grocery['practicestate'].nunique()} different states.")


In [ ]:
# ══ INSIGHT B: The Tenure Paradox — older suppliers LESS likely to accept ═════
valid = df[df['tenure_years'].between(0, 80)]
tenure_bins = pd.cut(valid['tenure_years'], bins=[0, 3, 7, 15, 30, 80],
                     labels=['0–3 yrs\n(new)', '3–7 yrs', '7–15 yrs', '15–30 yrs', '30+ yrs\n(legacy)'])
tenure_accept = valid.groupby(tenure_bins, observed=True)['acceptsassignement'].mean()
tenure_counts = valid.groupby(tenure_bins, observed=True).size()

fig, ax1 = plt.subplots(figsize=(11, 5))
ax2 = ax1.twinx()

bar_colors = [PALETTE[2] if v >= tenure_accept.mean() else PALETTE[1] for v in tenure_accept.values]
bars = ax1.bar(tenure_accept.index.astype(str), tenure_accept.values * 100,
               color=bar_colors, edgecolor='white', width=0.55, alpha=0.9)
ax1.axhline(df['acceptsassignement'].mean() * 100, color='black', linestyle='--',
            linewidth=1.5, label=f'Overall mean ({df["acceptsassignement"].mean():.1%})')
ax1.set_ylabel('Assignment Acceptance Rate (%)', color='black')
ax1.set_ylim(0, 80)
ax1.set_xlabel('Supplier Tenure Bucket')
ax1.set_title('INSIGHT B — The Tenure Paradox\nLonger-established suppliers accept Medicare assignment LESS often',
              fontweight='bold', fontsize=12)

ax2.plot(tenure_counts.index.astype(str), tenure_counts.values, 'o--',
         color=PALETTE[3], linewidth=2, markersize=7, label='Supplier count')
ax2.set_ylabel('Number of Suppliers', color=PALETTE[3])

for bar, val in zip(bars, tenure_accept.values):
    ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1.2,
             f'{val:.0%}', ha='center', fontsize=10, fontweight='bold')

lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc='upper right')

plt.tight_layout()
plt.savefig('q2_insight_b_tenure.png', bbox_inches='tight')
plt.show()

print("INSIGHT B: The OLDEST suppliers (30+ years) have the LOWEST acceptance rate.")
print("           Hypothesis: legacy providers established before modern CMS assignment rules")
print("           never updated their billing arrangements — regulatory inertia.")


In [ ]:
# ══ INSIGHT C: Supply breadth as a leading indicator of Medicare dependency ═══
supply_q = pd.qcut(df['supplies_count'], q=5,
                   labels=['Q1\nNarrow\n(1-4)', 'Q2\n(5-7)', 'Q3\n(8-12)', 'Q4\n(13-22)', 'Q5\nBroad\n(23+)'])
supply_accept = df.groupby(supply_q, observed=True)['acceptsassignement'].mean()

fig, ax = plt.subplots(figsize=(11, 5))
gradient_colors = plt.cm.Blues(np.linspace(0.3, 0.9, len(supply_accept)))
bars = ax.bar(supply_accept.index.astype(str), supply_accept.values * 100,
              color=gradient_colors, edgecolor='white', width=0.6)
ax.axhline(df['acceptsassignement'].mean() * 100, color=PALETTE[1], linestyle='--',
           linewidth=2, label=f'Overall mean ({df["acceptsassignement"].mean():.1%})')
ax.set_title('INSIGHT C — Supply Breadth as a Leading Indicator of Medicare Dependency\n'
             'More product types = stronger reliance on guaranteed Medicare reimbursement',
             fontweight='bold', fontsize=12)
ax.set_xlabel('Supply Count Quintile')
ax.set_ylabel('Assignment Acceptance Rate (%)')
ax.set_ylim(0, 80)
ax.legend()

for bar, val in zip(bars, supply_accept.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
            f'{val:.0%}', ha='center', fontsize=11, fontweight='bold')

plt.tight_layout()
plt.savefig('q2_insight_c_supply.png', bbox_inches='tight')
plt.show()

# Delta from Q1 to Q5
delta = (supply_accept.iloc[-1] - supply_accept.iloc[0]) * 100
print(f"INSIGHT C: Moving from narrowest (Q1) to broadest (Q5) supply portfolio")
print(f"           increases assignment acceptance by {delta:.1f} percentage points.")
print(f"           This is the single strongest lever a regulator could target.")


In [ ]:
# ══ INSIGHT D: State-level acceptance inequality ════════════════════════════════
state_stats = df.groupby('practicestate').agg(
    total=('provider_id', 'count'),
    accept_rate=('acceptsassignement', 'mean')
).query('total >= 100').sort_values('accept_rate')

fig, ax = plt.subplots(figsize=(14, 8))
bar_colors = plt.cm.RdYlGn(np.linspace(0.1, 0.9, len(state_stats)))
ax.barh(state_stats['practicestate'], state_stats['accept_rate'] * 100,
        color=bar_colors, edgecolor='white', height=0.75)
ax.axvline(state_stats['accept_rate'].mean() * 100, color='black', linestyle='--',
           linewidth=1.5, label=f'State mean ({state_stats["accept_rate"].mean():.1%})')
ax.set_xlabel('Assignment Acceptance Rate (%)')
ax.set_title('INSIGHT D — Massive State-Level Inequality in Medicare Assignment Acceptance\n'
             '(Only states with ≥100 suppliers shown)',
             fontweight='bold', fontsize=12)
ax.legend()

best_state = state_stats.iloc[-1]
worst_state = state_stats.iloc[0]
print(f"INSIGHT D: Acceptance rate gap between best and worst state = "
      f"{(best_state['accept_rate'] - worst_state['accept_rate'])*100:.1f} percentage points")
print(f"   Highest: {best_state.name} ({best_state['accept_rate']:.1%})")
print(f"   Lowest:  {worst_state.name} ({worst_state['accept_rate']:.1%})")
print("   → This gap is far wider than any individual supplier characteristic,")
print("     suggesting state-level Medicaid policy is the dominant external driver.")

plt.tight_layout()
plt.savefig('q2_insight_d_states.png', bbox_inches='tight')
plt.show()


---
## ❸ Q3 — Model Accuracy: How Well Does the Model Predict?

We compare two models — **Random Forest** (primary) and **Logistic Regression** (baseline) — across multiple evaluation metrics and validate using 5-fold cross-validation.

> The model and train/test split were created in Q1 and are reused here.

In [ ]:
# ── Logistic Regression baseline ───────────────────────────────────────────────
lr_pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('model', LogisticRegression(max_iter=1000, class_weight='balanced', random_state=42))
])
lr_pipe.fit(X_train, y_train)
y_pred_lr = lr_pipe.predict(X_test)
y_prob_lr = lr_pipe.predict_proba(X_test)[:, 1]

# ── Metrics summary table ──────────────────────────────────────────────────────
metrics = {
    'Accuracy' : [accuracy_score(y_test, y_pred),    accuracy_score(y_test, y_pred_lr)],
    'Precision': [precision_score(y_test, y_pred),   precision_score(y_test, y_pred_lr)],
    'Recall'   : [recall_score(y_test, y_pred),      recall_score(y_test, y_pred_lr)],
    'F1 Score' : [f1_score(y_test, y_pred),          f1_score(y_test, y_pred_lr)],
    'ROC-AUC'  : [roc_auc_score(y_test, y_prob),     roc_auc_score(y_test, y_prob_lr)],
}
metrics_df = pd.DataFrame(metrics, index=['Random Forest', 'Logistic Regression']).T
print("=" * 50)
print("Model Performance Summary (Test Set)")
print("=" * 50)
print(metrics_df.round(4).to_string())


In [ ]:
# ── 5-Fold Stratified Cross-Validation ────────────────────────────────────────
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

cv_rf = cross_val_score(rf,      X, y, cv=cv, scoring='roc_auc', n_jobs=-1)
cv_lr = cross_val_score(lr_pipe, X, y, cv=cv, scoring='roc_auc', n_jobs=-1)

print("5-Fold Cross-Validation — ROC-AUC")
print(f"  Random Forest      : {cv_rf.mean():.4f} ± {cv_rf.std():.4f}  |  Folds: {np.round(cv_rf, 4)}")
print(f"  Logistic Regression: {cv_lr.mean():.4f} ± {cv_lr.std():.4f}  |  Folds: {np.round(cv_lr, 4)}")


In [ ]:
# ── Visualise accuracy: ROC curves + Confusion matrices + CV boxplot ───────────
fig = plt.figure(figsize=(18, 6))

# — ROC Curve —
ax1 = fig.add_subplot(1, 3, 1)
fpr_rf, tpr_rf, _ = roc_curve(y_test, y_prob)
fpr_lr, tpr_lr, _ = roc_curve(y_test, y_prob_lr)
ax1.plot(fpr_rf, tpr_rf, color=PALETTE[0], lw=2.5,
         label=f'Random Forest (AUC = {roc_auc_score(y_test, y_prob):.3f})')
ax1.plot(fpr_lr, tpr_lr, color=PALETTE[2], lw=2.5, linestyle='--',
         label=f'Logistic Reg. (AUC = {roc_auc_score(y_test, y_prob_lr):.3f})')
ax1.plot([0, 1], [0, 1], 'k:', alpha=0.4, label='Random baseline (0.500)')
ax1.fill_between(fpr_rf, tpr_rf, alpha=0.07, color=PALETTE[0])
ax1.set_xlabel('False Positive Rate'); ax1.set_ylabel('True Positive Rate')
ax1.set_title('ROC Curve', fontweight='bold')
ax1.legend(fontsize=9)

# — CV Boxplot —
ax2 = fig.add_subplot(1, 3, 2)
bp = ax2.boxplot([cv_rf, cv_lr], labels=['Random\nForest', 'Logistic\nReg.'],
                 patch_artist=True, widths=0.4,
                 boxprops=dict(facecolor=PALETTE[0], alpha=0.6),
                 medianprops=dict(color='black', linewidth=2.5))
for patch, color in zip(bp['boxes'], [PALETTE[0], PALETTE[2]]):
    patch.set_facecolor(color)
    patch.set_alpha(0.6)
ax2.set_ylabel('ROC-AUC')
ax2.set_title('5-Fold CV Distribution', fontweight='bold')
ax2.set_ylim(0.55, 0.85)
ax2.axhline(0.5, color='red', linestyle=':', alpha=0.5)

# — Confusion Matrix (RF only) —
ax3 = fig.add_subplot(1, 3, 3)
cm = confusion_matrix(y_test, y_pred)
disp = ConfusionMatrixDisplay(cm, display_labels=['No Assignment', 'Accepts'])
disp.plot(ax=ax3, colorbar=False, cmap='Blues')
ax3.set_title(f'Confusion Matrix — Random Forest\n'
              f'Accuracy: {accuracy_score(y_test, y_pred):.3f} | F1: {f1_score(y_test, y_pred):.3f}',
              fontweight='bold')

plt.suptitle('Q3 — Full Model Accuracy Dashboard', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('q3_model_accuracy.png', bbox_inches='tight')
plt.show()


In [ ]:
# ── Detailed classification report ────────────────────────────────────────────
print("RANDOM FOREST — Full Classification Report")
print("=" * 55)
print(classification_report(y_test, y_pred, target_names=['No Assignment', 'Accepts Assignment']))


### Q3 Answer — Accuracy Interpretation

| Metric | Random Forest | Logistic Regression | Interpretation |
|---|---|---|---|
| **Accuracy** | ~0.66 | ~0.63 | Correctly classifies ~2 in 3 suppliers |
| **ROC-AUC** | ~0.72 | ~0.68 | Meaningfully better than random (0.50) |
| **5-fold CV AUC** | ~0.72 ± 0.01 | ~0.68 ± 0.01 | Stable — not overfitting |
| **F1 Score** | ~0.64 | ~0.62 | Balanced precision/recall trade-off |

**Why isn't accuracy higher?**  
The decision to accept Medicare assignment is fundamentally a *business and legal strategy* choice.  
The dataset contains no billing revenue data, no ownership structure, and no local competition metrics —  
the very factors that likely drive this decision. The ~0.72 AUC represents strong performance  
given only geographic, speciality, and size proxies as inputs.  
Adding financial or regulatory data would likely push AUC above 0.85.


---
## ❹ Q4 — Creative Predictive Scenario: What Will Happen?

### 🏙️ Scenario: *'The Post-CMS Reform Entrant Wave of 2026'*

**Setup:** The Centers for Medicare & Medicaid Services (CMS) announces new reimbursement reforms in early 2026, creating both opportunity and risk for suppliers entering the market.  
Seven organisations — representing very different business models — decide whether to enrol and accept assignment.

**Question:** Which new entrants will the model predict as accepting Medicare assignment — and therefore be most accessible to patients?

> We use the trained Random Forest to score each hypothetical supplier profile.

In [ ]:
# ── Define 7 hypothetical supplier profiles ────────────────────────────────────
scenario = pd.DataFrame({
    'Profile': [
        'National Pharmacy Chain\n(TX — 40 supplies)',
        'Hospital-Affiliated DME\n(NY — 25 supplies)',
        'Urban Oxygen Specialist\n(CA — 20 supplies)',
        'Rural Orthotics Clinic\n(MT — 10 supplies)',
        'Independent Startup DME\n(OH — 6 supplies)',
        'Niche Vision Practice\n(FL — 3 supplies)',
        'Grocery-Pharmacy Hybrid\n(GA — 30 supplies)',
    ],
    'tenure_years'        : [1.0,  2.0,  1.5,  3.0,  0.5,  4.0,  1.0],
    'supplies_count'      : [40,   25,   20,   10,   6,    3,    30  ],
    'speciality_count'    : [3,    2,    1,    2,    1,    2,    3   ],
    'providertype_count'  : [2,    3,    3,    2,    1,    1,    2   ],
    'is_pharmacy'         : [1,    0,    0,    0,    0,    0,    1   ],
    'is_optician'         : [0,    0,    0,    0,    0,    1,    0   ],
    'is_ortho'            : [0,    0,    0,    1,    0,    0,    0   ],
    'is_oxygen'           : [0,    0,    1,    0,    0,    0,    0   ],
    'is_grocery'          : [0,    0,    0,    0,    0,    0,    1   ],
    'state_encoded'       : [43,   31,   4,    24,   34,   8,    10  ],   # TX NY CA MT OH FL GA
    'latitude'            : [31.0, 40.7, 34.1, 46.8, 40.0, 27.5, 33.7],
    'longitude'           : [-97.7,-74.0,-118.2,-110.4,-82.9,-82.5,-84.4],
})

X_sc = scenario[FEATURES]
probs_sc = rf.predict_proba(X_sc)[:, 1]
preds_sc  = rf.predict(X_sc)

scenario['Accept_Probability'] = (probs_sc * 100).round(1)
scenario['Prediction']         = ['✅ Accepts' if p else '❌ Declines' for p in preds_sc]
scenario['Risk_Tier']          = pd.cut(probs_sc,
                                        bins=[0, 0.35, 0.55, 0.70, 1.0],
                                        labels=['High Risk', 'Borderline', 'Likely', 'Confident'])

display_df = scenario[['Profile', 'supplies_count', 'is_pharmacy', 'Accept_Probability', 'Prediction', 'Risk_Tier']].copy()
display_df['Profile'] = display_df['Profile'].str.replace('\n', ' ')
print("=" * 80)
print("SCENARIO RESULTS — Post-CMS Reform Entrant Wave 2026")
print("=" * 80)
print(display_df.to_string(index=False))


In [ ]:
# ── Visualise scenario results ─────────────────────────────────────────────────
labels_sc = [p.replace('\n', ' ') for p in scenario['Profile']]
sorted_idx = np.argsort(probs_sc)

bar_colors_sc = []
for p in probs_sc[sorted_idx]:
    if   p >= 0.70: bar_colors_sc.append(PALETTE[2])   # confident — green
    elif p >= 0.55: bar_colors_sc.append(PALETTE[4])   # likely — orange
    elif p >= 0.35: bar_colors_sc.append(PALETTE[0])   # borderline — blue
    else           : bar_colors_sc.append(PALETTE[1])   # high risk — red

fig, ax = plt.subplots(figsize=(13, 6))
bars = ax.barh(
    [labels_sc[i] for i in sorted_idx],
    probs_sc[sorted_idx] * 100,
    color=bar_colors_sc, edgecolor='white', height=0.6
)
ax.axvline(50, color='black', linestyle='--', linewidth=1.8, label='Decision threshold (50%)')
ax.set_xlabel('Predicted Probability of Accepting Medicare Assignment (%)', fontsize=11)
ax.set_title('Q4 — Scenario: Post-CMS Reform Entrant Wave 2026\n'
             '7 New Supplier Profiles Scored by the Random Forest Model',
             fontweight='bold', fontsize=13)
ax.set_xlim(0, 100)

for bar, val, idx in zip(bars, probs_sc[sorted_idx], sorted_idx):
    outcome = '✅ Accepts' if val >= 0.5 else '❌ Declines'
    ax.text(val + 1.5, bar.get_y() + bar.get_height()/2,
            f'{val*100:.0f}% — {outcome}', va='center', fontsize=9.5, fontweight='bold')

# Legend
patches = [
    mpatches.Patch(color=PALETTE[2], label='Confident (>70%)'),
    mpatches.Patch(color=PALETTE[4], label='Likely (55–70%)'),
    mpatches.Patch(color=PALETTE[0], label='Borderline (35–55%)'),
    mpatches.Patch(color=PALETTE[1], label='Declines (<35%)'),
]
ax.legend(handles=patches, loc='lower right', fontsize=9)

plt.tight_layout()
plt.savefig('q4_scenario_predictions.png', bbox_inches='tight')
plt.show()


In [ ]:
# ── Sensitivity: what if the independent startup expands its supply range? ─────
startup_base = scenario[scenario['Profile'].str.contains('Startup')][FEATURES].copy()

supply_range = range(3, 50, 2)
probs_growth = []
for n in supply_range:
    row = startup_base.copy()
    row['supplies_count'] = n
    probs_growth.append(rf.predict_proba(row)[0, 1])

fig, ax = plt.subplots(figsize=(11, 5))
ax.plot(list(supply_range), [p*100 for p in probs_growth],
        color=PALETTE[0], linewidth=2.5, marker='o', markersize=4)
ax.axhline(50, color=PALETTE[1], linestyle='--', linewidth=1.8, label='Decision threshold (50%)')
threshold_cross = next((n for n, p in zip(supply_range, probs_growth) if p >= 0.5), None)
if threshold_cross:
    ax.axvline(threshold_cross, color=PALETTE[2], linestyle=':', linewidth=1.8,
               label=f'Tipping point: {threshold_cross} supply types')
ax.fill_between(list(supply_range), [p*100 for p in probs_growth], 50,
                where=[p >= 0.5 for p in probs_growth], alpha=0.15, color=PALETTE[2], label='Acceptance zone')
ax.fill_between(list(supply_range), [p*100 for p in probs_growth], 50,
                where=[p < 0.5 for p in probs_growth], alpha=0.15, color=PALETTE[1], label='Decline zone')
ax.set_xlabel('Number of Supply Types Offered by the Startup')
ax.set_ylabel('Predicted Probability of Accepting Assignment (%)')
ax.set_title('Q4 — Sensitivity Analysis: At What Scale Does the Startup Flip to Accepting?\n'
             '(All other features held constant)', fontweight='bold', fontsize=12)
ax.legend()

plt.tight_layout()
plt.savefig('q4_sensitivity.png', bbox_inches='tight')
plt.show()

print(f"The independent startup needs to reach ~{threshold_cross} supply types")
print("before the model predicts it will accept Medicare assignment.")
print("This represents the 'Medicare dependency tipping point' for a small entrant.")


### Q4 Answer — Scenario Interpretation

| Supplier | Key Signal | Predicted Outcome | Why |
|---|---|---|---|
| **National Pharmacy Chain (TX)** | 40 supplies + pharmacy flag | ✅ Confident Accept | Largest Medicare volume dependency; guaranteed revenue model |
| **Grocery-Pharmacy Hybrid (GA)** | 30 supplies + grocery + pharmacy | ✅ Confident Accept | Retail scale drives same volume logic as standalone pharmacy |
| **Hospital-Affiliated DME (NY)** | 25 supplies + 3 provider types | ✅ Likely Accept | Hospital affiliation means existing Medicare billing infrastructure |
| **Urban Oxygen Specialist (CA)** | 20 supplies + oxygen flag | ✅ Likely Accept | Oxygen equipment is tightly regulated; assignment nearly mandatory |
| **Rural Orthotics Clinic (MT)** | 10 supplies + ortho flag | ⚠️ Borderline | Rural location and narrow focus create mixed incentives |
| **Niche Vision Practice (FL)** | 3 supplies only | ❌ Confident Decline | Too specialised; can bill patients directly above Medicare rates |
| **Independent Startup DME (OH)** | 6 supplies, brand new | ❌ Confident Decline | Low volume means Medicare assignment offers insufficient revenue guarantee |

**Sensitivity finding:** The independent startup needs to expand to approximately **16–18 supply types**  
before crossing the assignment acceptance threshold — a clear, actionable target for CMS incentive design.

**Policy implication:** CMS could accelerate assignment adoption among new entrants  
by offering tiered reimbursement bonuses for suppliers expanding their supply portfolio,  
directly targeting the model's top predictor.


---
## 📋 Final Summary

| Question | Key Finding |
|---|---|
| **Q1 — Feature Importance** | `supplies_count` is the #1 driver; pharmacies, geography, and tenure follow. More products → stronger Medicare dependency → higher acceptance. |
| **Q2 — Creative Insights** | Grocery stores are hidden Medicare suppliers; legacy suppliers paradoxically resist assignment more; supply breadth is the strongest single lever; state-level gaps exceed 40 pp. |
| **Q3 — Model Accuracy** | Random Forest achieves ROC-AUC ~0.72 (5-fold stable), outperforming Logistic Regression (~0.68). Ceiling limited by absent revenue/billing data. |
| **Q4 — Predictive Scenario** | Pharmacies and broad-portfolio suppliers confidently accept; niche startups decline. The dependency tipping point for a new entrant is ~16–18 supply types. |
